In [1]:
print("Projeto Churn: notebook funcionando!")

Projeto Churn: notebook funcionando!


In [2]:
import pandas as pd

print(pd.__version__)

3.1.1


In [3]:
url = (
    "https://raw.githubusercontent.com/IBM/"
    "telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
)

df = pd.read_csv(url)

df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.shape

(7043, 21)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [6]:
total_numerico = pd.to_numeric(df["TotalCharges"], errors="coerce")

problemas = total_numerico.isna()

df.loc[problemas, ["customerID", "tenure", "TotalCharges"]]

,customerID,tenure,TotalCharges
488,4472-LVYGI,0,
753,3115-CZMZD,0,
936,5709-LVOEQ,0,
1082,4367-NUYAO,0,
1340,1371-DWPAZ,0,
3331,7644-OMVMY,0,
3826,3213-VVOLG,0,
4380,2520-SGTTA,0,
5218,2923-ARZLG,0,
6670,4075-WKNIU,0,


In [7]:
df.loc[problemas, "TotalCharges"].map(repr)

488     ' '
753     ' '
936     ' '
1082    ' '
1340    ' '
3331    ' '
3826    ' '
4380    ' '
5218    ' '
6670    ' '
6754    ' '
Name: TotalCharges, dtype: str

In [8]:
exemplo = pd.Series([True, False, True])
exemplo.sum()

np.int64(2)

In [9]:
problemas.sum()

np.int64(11)

In [10]:
df.loc[problemas, "tenure"]

488     0
753     0
936     0
1082    0
1340    0
3331    0
3826    0
4380    0
5218    0
6670    0
6754    0
Name: tenure, dtype: int64

In [11]:
df.loc[problemas, "customerID"]

488     4472-LVYGI
753     3115-CZMZD
936     5709-LVOEQ
1082    4367-NUYAO
1340    1371-DWPAZ
3331    7644-OMVMY
3826    3213-VVOLG
4380    2520-SGTTA
5218    2923-ARZLG
6670    4075-WKNIU
6754    2775-SEFEE
Name: customerID, dtype: str

In [12]:
df["Churn"].value_counts(dropna=False)

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [13]:
df["Churn"].value_counts(normalize=True, dropna=False)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

In [14]:
df["customerID"].duplicated().sum()

np.int64(0)

In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

## Diagnóstico inicial dos dados

- A base possui 7.043 registros e 21 colunas.
- Não foram encontrados IDs repetidos nem linhas inteiramente duplicadas.
- O pandas não reconheceu valores ausentes na tabela original.
- TotalCharges foi carregada como texto e contém 11 valores formados por um espaço.
- Os 11 registros com esse problema apresentam tenure igual a zero.
- Aproximadamente 26,5% dos clientes da base cancelaram.
- Nenhum registro foi removido e nenhuma coluna original foi alterada.

### Decisão
Investigar e definir o tratamento de TotalCharges antes da modelagem.
Não substituir os espaços por zero sem justificativa.

In [17]:
from sklearn.model_selection import train_test_split

exemplo = [10, 20, 30, 40, 50]

parte_treino, parte_reservada = train_test_split(
    exemplo,
    test_size=0.2,
    random_state=42
)

print("Treino:", parte_treino)
print("Reservada:", parte_reservada)

Treino: [50, 30, 10, 40]
Reservada: [20]


In [18]:
desenvolvimento, teste = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Churn"]
)

print("Desenvolvimento:", desenvolvimento.shape)
print("Teste:", teste.shape)

Desenvolvimento: (5634, 21)
Teste: (1409, 21)


In [19]:
treino, validacao = train_test_split(
    desenvolvimento,
    test_size=0.25,
    random_state=42,
stratify=desenvolvimento["Churn"])

print("Treino:", treino.shape)
print("Validação:", validacao.shape)

Treino: (4225, 21)
Validação: (1409, 21)


In [20]:
print("Base original:", len(df))
print("Treino:", len(treino))
print("Validação:", len(validacao))
print("Teste:", len(teste))

Base original: 7043
Treino: 4225
Validação: 1409
Teste: 1409


In [21]:
print(treino["Churn"].value_counts(normalize=True, dropna=False))
print(validacao["Churn"].value_counts(normalize=True, dropna=False))

Churn
No     0.734675
Yes    0.265325
Name: proportion, dtype: float64
Churn
No     0.734564
Yes    0.265436
Name: proportion, dtype: float64


In [22]:
ids_treino = set(treino["customerID"])
ids_validacao = set(validacao["customerID"])

print("Clientes em comum:", len(ids_treino.intersection(ids_validacao)))

Clientes em comum: 0


In [23]:
ids_teste = set(teste["customerID"])
print("Validação e teste:", len(ids_validacao.intersection(ids_teste)))
print("Treino e teste:", len(ids_treino.intersection(ids_teste)))

Validação e teste: 0
Treino e teste: 0


## Separação dos dados

- Treino: 4.225 clientes, aproximadamente 60% da base.
- Validação: 1.409 clientes, aproximadamente 20% da base.
- Teste: 1.409 clientes, aproximadamente 20% da base.
- A divisão foi estratificada pela coluna Churn, com random_state=42.
- Não há clientes em comum entre os conjuntos.
- Treino e validação apresentam aproximadamente 26,5% de cancelamentos.

### Regra de uso
O treino será utilizado para ajustar os modelos e o pré-processamento.
A validação será utilizada para comparar alternativas e escolher configurações.
O teste ficará reservado para a avaliação final, sem orientar essas escolhas.

In [24]:
treino["Contract"].value_counts(dropna=False)

Contract
Month-to-month    2326
Two year          1034
One year           865
Name: count, dtype: int64

In [27]:
pd.crosstab(
    treino["Contract"],
    treino["Churn"],
    normalize="index"
)

Churn,No,Yes
Contract,,
Month-to-month,0.569218,0.430782
One year,0.892486,0.107514
Two year,0.974855,0.025145


### Cancelamento por tipo de contrato

No conjunto de treino, a proporção de cancelamentos foi de 43,1% nos contratos mensais, 10,8% nos anuais e 2,5% nos bienais.

Essa associação não demonstra causalidade; diferenças no perfil dos clientes podem contribuir para o resultado.

In [28]:
treino["tenure"].describe()

count    4225.000000
mean       32.294438
std        24.667945
min         0.000000
25%         9.000000
50%        29.000000
75%        56.000000
max        72.000000
Name: tenure, dtype: float64

In [29]:
treino.groupby("Churn")["tenure"].describe()

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
No,3104.0,37.521907,24.269930,0.0,15.0,37.0,61.0,72.0
Yes,1121.0,17.819804,19.431418,1.0,2.0,10.0,29.0,72.0


### Cancelamento e tempo de relacionamento

No treino, a mediana do tempo de relacionamento é de 37 meses entre os clientes que não cancelaram e de 10 meses entre os que cancelaram.

Essas medianas representam o centro dos tempos ordenados de cada grupo. Não significam que todos os cancelamentos ocorreram aos 10 meses.

Para calcular a taxa de cancelamento entre clientes com até 10 meses, precisamos dividir o número de cancelamentos nessa faixa pelo total de clientes da mesma faixa.

A diferença observada entre as medianas não demonstra causalidade.

In [30]:
clientes_ate_10 = treino.loc[treino["tenure"] <= 10]

print("Clientes com até 10 meses:", len(clientes_ate_10))

Clientes com até 10 meses: 1181


In [32]:
clientes_ate_10["Churn"].value_counts(dropna=False)

Churn
No     598
Yes    583
Name: count, dtype: int64

In [33]:
clientes_ate_10["Churn"].value_counts(normalize=True, dropna=False)

Churn
No     0.506351
Yes    0.493649
Name: proportion, dtype: float64

### Cancelamento entre clientes com até 10 meses

No treino, entre os 1.181 clientes com até 10 meses de relacionamento, 583 cancelaram, aproximadamente 49,4% desse grupo.

Essa taxa descreve o grupo analisado. Não demonstra que o menor tempo de relacionamento cause cancelamento.

In [35]:
clientes_mais_10 = treino.loc[treino["tenure"] > 10]
print("Clientes com mais de 10 meses:", len(clientes_mais_10))

Clientes com mais de 10 meses: 3044


In [36]:
clientes_mais_10["Churn"].value_counts(normalize=True, dropna=False)

Churn
No     0.823259
Yes    0.176741
Name: proportion, dtype: float64

### Cancelamento entre clientes com mais de 10 meses

No treino, a taxa de cancelamento foi de 17,7% entre os 3.044 clientes com mais de 10 meses, contra 49,4% entre os 1.181 clientes com até 10 meses — diferença de aproximadamente 31,7 pontos percentuais.

O corte de 10 meses foi escolhido de forma exploratória, com base na mediana do grupo que cancelou. Essa associação não demonstra que menor tempo de relacionamento cause cancelamento.